# import pandas as pd
import numpy as np
import hdf5storage

from matplotlib import pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.svm import SVC
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# from xgboost import XGBClassifier

rcParams = {'figure.figsize': (15 ,8)}

sns.set(rc=rcParams)

**1- Business Problem**

O problema de neg?cio consise na classifica??o de tipo de faltas pelo seu padrao de ocorr?ncia

Geralmente em um sistema de energia eletrica a falta ocorre de duas maneiras, simetrica ou assimetrica. Dessa forma,
podemos ter faltas, nas tr?s fases A, B, C e para a terra.

- Faltas Sim?tricas: Tipo de falta que ocorre quando h? uma falta - L-L-L ou L-L-L-G;
- Faltas Assim?tricas: Tipo L-G, L-L, L-L-G

**Hipoteses**: 

- **1- Correntes muito altas tendem a ter tido uma falta**
- **2- Em uma falta h? um afundamento de tens?o**

**2-Coleta**

In [ ]:
files = [
    "ia_a_g.mat", "ib_a_g.mat", "ic_a_g.mat", "va_a_g.mat", "vb_a_g.mat", "vc_a_g.mat",
    "ia_b_g.mat", "ib_b_g.mat", "ic_b_g.mat", "va_b_g.mat", "vb_b_g.mat", "vc_b_g.mat",
    "ia_c_g.mat", "ib_c_g.mat", "ic_c_g.mat", "va_c_g.mat", "vb_c_g.mat", "vc_c_g.mat",
    "ia_tri_g.mat", "ib_tri_g.mat", "ic_tri_g.mat", "va_tri_g.mat", "vb_tri_g.mat", "vc_tri_g.mat"
]

In [ ]:
import hdf5storage
mat = hdf5storage.loadmat('test.mat')

In [ ]:
print("reading data")
for i, file in enumerate(files):
    mat = hdf5storage.loadmat('test.mat')["ans"]
    if i == 0:
        time = mat[0]
        value = mat[1]
        train_df = pd.DataFrame({"time": time, file: value})
    else:
        train_df[file] = value

print("finished")
df

In [ ]:
print("Cria a label")
time_fault = 0.1

train_df["is_fault"] = 0
train_df.loc[df.time >= 0.1, "is_fault"] = 1
train_df["is_fault"]

**Conjunto de Dados**:
    - IA/IB/IC-Va/Vb/Vc - Correntes tensoes em cada fase (linha)
    - Output (S): faz a marcacao se houve falta.

**3-Preprocess**

In [ ]:
# Nulos
train_df.isnull().sum()

Sem valores nulos nas colunas de corrente e tensao. Colunas 7 e 8 muitos contem todos valores nulos.

In [ ]:
# Metricas Gerais
train_df.describe()

Minimo e maximo de corrente parecem estar muito longe da media ou da mediana o que pode indicar falta

In [ ]:
fault_class = train_df["is_fault"].value_counts().reset_index(
                ).rename(columns={"index":"class", "is_fault": "qnt"})


sns.barplot(data = fault_class, x="class", y="qnt")

**Correlacao com a Corrente**

In [ ]:
# Destribuicao de Corrente no conjunto de dados:
    
current_cols = [i for i in train_df.columns if "i" in i]

for current_col in current_cols:
    sns.displot(data=train_df, x=current_col, hue="is_fault", height=7, aspect=2)

In [ ]:
print("O maximo de corrente sem falta, que existe nos dados, em cada linha, foi:")

for current_col in current_cols:
    print(current_col, train_df.loc[train_df["is_fault"] == 0 ,current_col].max())

**Hipotese 1** - Claramente vemos que nossa hipotese 1 se mostra verdadeira para uma zona de altas correntes.
Porem para baixas correntes isso nao eh verdade. Algumas faltas ocorrem mesmo com baixas correntes em algumas linhas.
Isso pode ser, por uma falta em uma linha n?o afetar o funcionamento da corrente na outra que nao sofreu a falta.

In [ ]:
curr_cols = [i for i in train_df.columns if "i" in i]
train_df[curr_cols + ["fault"]].corr()["fault"]

Vemos que claramente conseguimos tra?ar zonas de subdivisao, porem como a correlacao nao eh linear o indice de correlacao sera
baixo

Utilizando a correlacao com os valores modulares, vamos conseguir ver melhor essa correlacao

In [ ]:
# Deprecated: O modelo se mostrou ... TODO: Resultado
# Criacao de variaveis modulares de corrente

# curr_cols = [i for i in train_df.columns if "I" in i]
# print(curr_cols)
# train_df[curr_cols + ["Output (S)"]].corr()["Output (S)"]

# for curr_col in curr_cols:
#     train_df[curr_col + "_MOD"] = train_df[curr_col].abs()
    
# for curr_col in curr_cols:
#     sns.displot(data=train_df, x=curr_col, hue="Output (S)", height=7, aspect=2)
    
# print(train_df[curr_cols + ["Output (S)"]].corr()["Output (S)"])

Ok, parece que agora, conseguimos ver que a corrente tem correlacao com a falta pelo indice de correlacao.

Neste ponto ja conseguimos criar uma regra de separacao e testar uma modelagem em cima desses dados e criar outras features.

Porem, ainda temos algumns pontos de falta que nao conseguimos classificar muito bem.
Vamos verificar se pela tensao conseguimos utilizar esses pontos para classificao.

**Consideramos a hipotese 1 validada para correntes altas, temos faltas. Porem precisamos entender se conseguimos explicar os casos de correntes baixas**

Qual a corrente de carga do circuito?

**Correlacao com a Tensao**

In [ ]:
# Destribuicao de Corrente no conjunto de dados:
    
curr_cols = [i for i in train_df.columns if "v" in i]

for curr_col in curr_cols:
    sns.displot(data=train_df, x=curr_col, hue="fault", height=7, aspect=2)

Para a tensao **a hipotese 2 parece tambem estar confirmada, como esperavamos, em casos de falta, temos "afundamentos" de tensao.** Isso acontece porque em uma falta de energia, geralmente tendemos a esperar que algo tenha entrado em contato com a linha, como uma arvore encostando na linha de transmissao ou um ganho caindo.

Dessa forma ao entrar em contato com um ponto de tensao 0 V, h? um afundamento de tens?o.

**Classificao de Padroes e Faltas**

Para a classificao, vamos usar 3 metodos distintos:

- 1- M?quina de Aprendizes Keras
- 2- Trein?-las utilizando fun??es de perda do tipo regress?o (como MSE por exemplo)
- 3- Trein?-las utilizando fun??es de perda probabil?sticas (Cross-entropy, KLD ou KL);
- 4- Construir modelos aut?nomos bayesianos;
- 5- Comparar as m?quinas com respeito ao desempenho;

In [ ]:
train_df[["ia", "ib", "ic"]].quantile(0.8)

In [ ]:
# Modelo baseline. A partir de uma corrente maior que o terceiro quartil clasificamos como falta

base = train_df[["ia", "ib", "ic"]].abs().quantile(0.4).sum()
print("fault detection baseline current:", base)
train_df["predict"] = np.where((train_df.ia.abs() + train_df.ib.abs() + train_df.ic.abs())/3 > base, 1, 0)
print((train_df["predict"] - train_df["fault"]).value_counts())
train_df = train_df.drop(columns="predict")

In [ ]:
train_df.sample()

## Prepara o conjunto para os algoritmos

In [ ]:
# Separe Dataset
test_size=0.2

features = train_df.drop(columns="fault")
target = train_df["fault"]

# Train and Validation set

# Deprecates: Using train_test_split
# num_val_samples = int(len(features) * 0.2)
# train_features = features[:-num_val_samples]
# train_target = target[:-num_val_samples]
# val_features = features[-num_val_samples:]
# val_target = target[-num_val_samples:]


train_features, val_features, train_target, val_target = train_test_split(features, target, test_size=test_size)

print("Number of traning samples", len(train_features))
print("Number of validation samples", len(val_features))
print("Number of features", train_features.shape[1], train_features.columns)

## Modelagem

**Tensor Flow**

In [ ]:
# Installing Tensorflow
# import sys
# !{sys.executable} -m pip install tensorflow
from tensorflow import keras
from keras.callbacks import History 

In [ ]:
# Prepare train and validation set - Tensorflow needs arrays
train_features, val_features = train_features.values, val_features.values
train_target, val_target = train_target.values.reshape(-1, 1), val_target.values.reshape(-1, 1)

In [ ]:
# analyze class imbalance
class_train_count = np.bincount(train_target[:, 0])
class_valid_count = np.bincount(val_target[:, 0])

print(train_target[0, 0])
print(counts)
print("Quantidade de faltas no dataset de treino", class_train_count[1])
print( round(class_train_count[1] / (class_train_count[0] + class_train_count[1])*100, 1 ), "%" )
print("Quantidade de faltas no dataset de validacao", class_valid_count[1])
print( round(class_valid_count[1] / (class_valid_count[0] + class_valid_count[1])*100, 1 ), "%" )

In [ ]:
# Normalize the data using traning set statistics

mean = np.mean(train_features, axis=0)
train_features -= mean
val_features -= mean
std = np.std(train_features, axis=0)
train_features /= std
val_features /= std

print("train_features", train_features)
print("val_features", val_features)
print("train_target", train_target)
print("val_target", val_target)

In [ ]:
# Criacao do modelo no Keras
# Camada Densa: foram criadas camadas densas. As camadas densas sao camadas totalmente conectadas. Ou seja, todos os neuronios
# de uma camada se conectam com todos os neuronios das camadas seguintes
# Dropout: o dropout eh uma tecnica de regularizacao utilizada para reduzir o sobreajuste em redes neurais
model = keras.Sequential(

    [
        keras.layers.Dense(
            256, activation="relu", input_shape=(train_features.shape[-1] ,)
        ),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid")
    ]
)

model.summary()

In [ ]:
metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(optimizer=keras.optimizers.Adam(1e-2), loss="MeanSquaredError", metrics=metrics)

model.fit(
    train_features,
    train_target,
    batch_size=2048,
    epochs=30,
    verbose=2,
    validation_data=(val_features, val_target)
)

**Treinando com uma funcao de loss de crossentropia**

Na teoria da informacao a entropia cruzada se refere ? diferen?a entre duas distribui??es de probabilidade p (verdadeira) e q (estimada) sobre o mesmo conjunto de eventos. Na pr?tica a entropia cruzada mede o n?mero de bits necess?rios para identificar um evento.


In [ ]:
# Treinando com uma funcao de loss de crossentropia
metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=metrics)

model.fit(
    train_features,
    train_target,
    batch_size=2048,
    epochs=30,
    verbose=2,
    validation_data=(val_features, val_target)
)

In [ ]:
metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(optimizer=keras.optimizers.Adam(1e-2), loss=keras.losses.KLDivergence(), metrics=metrics)

model.fit(
    train_features,
    train_target,
    batch_size=2048,
    epochs=30,
    verbose=2,
    validation_data=(val_features, val_target)
)

In [ ]:
metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(optimizer=keras.optimizers.Adam(1e-2), loss=keras.losses.BinaryCrossentropy(), metrics=metrics)

model.fit(
    train_features,
    train_target,
    batch_size=2048,
    epochs=30,
    verbose=2,
    validation_data=(val_features, val_target)
)

In [ ]:
# Cross validation and final results
scores = cross_val_score(model, X, y, cv=5)
print("scores", scores, "mean", scores.mean(), "std", scores.std() )

## Arvore de Decisao - Random Forest

In [ ]:
features

In [ ]:
# Decision Trees
train_features, val_features, train_target, val_target = train_test_split(features, target, test_size=0.2)

print("Number of traning samples", len(train_features))
print("Number of validation samples", len(val_features))
print("Number of features", train_features.shape[1])

In [ ]:
model = RandomForestClassifier(max_depth=2, random_state=0)
model.fit(train_features, train_target)
predict = model.predict(val_features)

In [ ]:
val_features["target"] = val_target
val_features["predict"] = predict

In [ ]:
def confusion_matrix(df: pd.DataFrame, col1: str, col2: str):
    """
    Given a dataframe with at least
    two categorical columns, create a 
    confusion matrix of the count of the columns
    cross-counts
    
    use like:
    
    >>> confusion_matrix(test_df, 'actual_label', 'predicted_label')
    """
    return (
            df
            .groupby([col1, col2])
            .size()
            .unstack(fill_value=0)
            )

In [ ]:
# matriz de confusao
conf_df = confusion_matrix(val_features, "target", "predict")
acuracia = (conf_df[0][0] + conf_df[1][1]) / conf_df.sum().sum()
print("acuracia", acuracia)
conf_df

In [ ]:
for f in val_features.columns:
    sns.displot(val_features, x=f, hue="predict", height=7, aspect=2)

In [ ]:
# Wrong predicitions
mistakes_df = val_features[val_features["target"] != val_features["predict"]]# Wrong predicitions
mistakes_df = val_features[val_features["target"] != val_features["predict"]]

for f in val_features.columns:
    sns.displot(mistakes_df, x=f, hue="predict", height=7, aspect=2)

In [ ]:
model.feature_importances_, val_features.columns

## Logistic Regression

In [ ]:
train_df.columns

In [ ]:
X = train_df.drop(columns='Output (S)')
y = train_df["Output (S)"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape,  y_train.shape, X_test.shape, y_test.shape

In [ ]:
model = LogisticRegression(random_state=0).fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
rmse(y_test, y_pred)

In [ ]:
model.feature_selection(

## Support Vector Machine

In [ ]:
model = SVC(gamma='auto')